# Notebook 05 — Model Comparison

Final test-set evaluation comparing three approaches to axle detection on the B-WIM bridge strain dataset:

| Method | Architecture | Description |
|---|---|---|
| **Baseline** | Signal processing | Savitzky-Golay smooth → `find_peaks` |
| **CNN** | 1D U-Net | Encoder/decoder with skip connections |
| **TCN** | Temporal Conv. Net | Dilated residual blocks, receptive field ≥ 1300 samples |

**Test set:** 3,215 vehicles (10 % held-out split, seed 42). Evaluation uses a ±5 sample tolerance window.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.dataset import build_datasets
from src.models import AxleUNet, AxleTCN
from src.baseline import signal_to_pulse_peaks
from src.evaluate import axle_level_metrics, evaluate_model

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

JSON_PATH = '../axle_data.json/axle_data.json'
CKPT_DIR  = '../checkpoints'
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TOL       = 5
SEED      = 42

print(f'Device: {DEVICE}')


In [ ]:
# Load datasets (uses the same train/val/test split as training)
train_ds, val_ds, test_ds = build_datasets(JSON_PATH, seed=SEED)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

print(f'Test set: {len(test_ds):,} vehicles')


In [ ]:
# Load trained model checkpoints
cnn_model = AxleUNet().to(DEVICE)
cnn_ckpt  = torch.load(os.path.join(CKPT_DIR, 'cnn_best.pt'), map_location=DEVICE, weights_only=False)
cnn_model.load_state_dict(cnn_ckpt['state_dict'])

tcn_model = AxleTCN().to(DEVICE)
tcn_ckpt  = torch.load(os.path.join(CKPT_DIR, 'tcn_best.pt'), map_location=DEVICE, weights_only=False)
tcn_model.load_state_dict(tcn_ckpt['state_dict'])

print(f"CNN best checkpoint: epoch {cnn_ckpt['epoch']}, val F1 = {cnn_ckpt['val_f1']:.4f}")
print(f"TCN best checkpoint: epoch {tcn_ckpt['epoch']}, val F1 = {tcn_ckpt['val_f1']:.4f}")


## Test-set Evaluation

All three methods are evaluated at **tolerance = 5 samples** (±5 around each ground-truth axle position).
A prediction within this window counts as a True Positive; excess predictions are False Positives; unmatched ground-truth axles are False Negatives.


In [ ]:
# Run evaluation for all three methods
test_signals = [test_ds[i][0].squeeze(0).numpy() for i in range(len(test_ds))]
test_pulses  = [test_ds[i][1].numpy() for i in range(len(test_ds))]

pred_base    = [signal_to_pulse_peaks(s) for s in test_signals]
base_metrics = axle_level_metrics(test_pulses, pred_base, tolerance=TOL)
cnn_metrics  = evaluate_model(cnn_model, test_loader, DEVICE, tolerance=TOL)
tcn_metrics  = evaluate_model(tcn_model, test_loader, DEVICE, tolerance=TOL)

# Build summary DataFrame
rows = []
for name, m in [('Peak Detection (baseline)', base_metrics),
                ('1D U-Net CNN',              cnn_metrics),
                ('TCN',                       tcn_metrics)]:
    rows.append({
        'Method':    name,
        'Precision': round(m['precision'], 4),
        'Recall':    round(m['recall'],    4),
        'F1':        round(m['f1'],        4),
        'MATE':      round(m['mate'],      3),
        'TP':        m['tp'],
        'FP':        m['fp'],
        'FN':        m['fn'],
    })

df = pd.DataFrame(rows).set_index('Method')
df.style.format({'Precision': '{:.4f}', 'Recall': '{:.4f}', 'F1': '{:.4f}', 'MATE': '{:.3f}'}) \
        .highlight_max(subset=['F1', 'Recall'], color='#d4edda') \
        .highlight_min(subset=['MATE', 'FN'],   color='#d4edda')


## Visual Comparison

### Precision / Recall / F1 / MATE


In [ ]:
labels       = ['Baseline', 'CNN', 'TCN']
metrics_list = [base_metrics, cnn_metrics, tcn_metrics]
colours      = ['steelblue', 'darkorange', 'seagreen']
x     = np.arange(len(labels))
width = 0.22

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scores panel
bar_prec = axes[0].bar(x - width, [m['precision'] for m in metrics_list], width, label='Precision', color=colours, alpha=0.75)
bar_rec  = axes[0].bar(x,          [m['recall']    for m in metrics_list], width, label='Recall',    color=colours, alpha=0.92)
bar_f1   = axes[0].bar(x + width,  [m['f1']        for m in metrics_list], width, label='F1',        color=colours, alpha=1.0)

for bars in (bar_prec, bar_rec, bar_f1):
    for bar in bars:
        v = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width() / 2, v + 0.002,
                     f'{v:.4f}', ha='center', va='bottom', fontsize=6.5, rotation=90)

axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
axes[0].set_ylim(0.85, 1.07)
axes[0].set_ylabel('Score')
axes[0].set_title(f'Precision / Recall / F1  (tolerance = {TOL} samples)')
axes[0].legend()

# MATE panel
mate_vals  = [m['mate'] for m in metrics_list]
bars_mate  = axes[1].bar(labels, mate_vals, color=colours)
for bar, v in zip(bars_mate, mate_vals):
    axes[1].text(bar.get_x() + bar.get_width() / 2, v + 0.01,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Samples')
axes[1].set_title('Mean Absolute Timing Error (MATE)')
axes[1].set_ylim(0, max(mate_vals) * 1.35)

plt.tight_layout()
plt.show()


### F1 Score vs Matching Tolerance

How does F1 degrade as the allowed timing window tightens?


In [ ]:
tolerances = [1, 2, 3, 5, 8, 10, 15, 20]
tol_results = {'Baseline': [], 'CNN': [], 'TCN': []}

for tol in tolerances:
    tol_results['Baseline'].append(axle_level_metrics(test_pulses, pred_base, tolerance=tol)['f1'])
    tol_results['CNN'].append(evaluate_model(cnn_model, test_loader, DEVICE, tolerance=tol)['f1'])
    tol_results['TCN'].append(evaluate_model(tcn_model, test_loader, DEVICE, tolerance=tol)['f1'])

line_styles = {
    'Baseline': dict(color='steelblue',  linestyle='--', marker='s', linewidth=2),
    'CNN':      dict(color='darkorange', linestyle='-',  marker='o', linewidth=2),
    'TCN':      dict(color='seagreen',   linestyle='-',  marker='^', linewidth=2),
}

fig, ax = plt.subplots(figsize=(8, 4))
for name, f1s in tol_results.items():
    ax.plot(tolerances, f1s, label=name, **line_styles[name])
ax.set_xlabel('Tolerance (samples)')
ax.set_ylabel('F1 Score')
ax.set_title('F1 Score vs Matching Tolerance')
ax.set_ylim(0.70, 1.02)
ax.legend()
ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


### Training Dynamics


In [ ]:
import csv

def read_log(path):
    epochs, train_l, val_l, val_f1 = [], [], [], []
    with open(path) as f:
        for row in csv.DictReader(f):
            epochs.append(int(row['epoch']))
            train_l.append(float(row['train_loss']))
            val_l.append(float(row['val_loss']))
            val_f1.append(float(row['val_f1']))
    return epochs, train_l, val_l, val_f1

cnn_e, cnn_tl, cnn_vl, cnn_vf1 = read_log(os.path.join(CKPT_DIR, 'cnn_log.csv'))
tcn_e, tcn_tl, tcn_vl, tcn_vf1 = read_log(os.path.join(CKPT_DIR, 'tcn_log.csv'))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(cnn_e, cnn_vl, label='CNN val loss', color='darkorange', linewidth=1.8)
ax1.plot(tcn_e, tcn_vl, label='TCN val loss', color='seagreen',   linewidth=1.8)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE Loss')
ax1.set_title('Validation Loss over Training')
ax1.legend(); ax1.grid(True, linestyle=':', alpha=0.5)

ax2.plot(cnn_e, cnn_vf1, label='CNN val F1', color='darkorange', marker='.', linewidth=1.8)
ax2.plot(tcn_e, tcn_vf1, label='TCN val F1', color='seagreen',   marker='.', linewidth=1.8)
# mark best epochs
cnn_best_ep = cnn_e[int(np.argmax(cnn_vf1))]
tcn_best_ep = tcn_e[int(np.argmax(tcn_vf1))]
ax2.axvline(cnn_best_ep, color='darkorange', linestyle=':', alpha=0.7, label=f'CNN best (ep {cnn_best_ep})')
ax2.axvline(tcn_best_ep, color='seagreen',   linestyle=':', alpha=0.7, label=f'TCN best (ep {tcn_best_ep})')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('F1 Score')
ax2.set_title('Validation F1 over Training')
ax2.legend(fontsize=9); ax2.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()


## Key Findings

| Finding | Detail |
|---|---|
| **TCN achieves perfect recall** | Zero missed axles on the entire test set (14,574 axles across 3,215 vehicles) |
| **Baseline misses 11.6 % of axles** | 1,686 false negatives — catastrophic for GVW calculation in a real B-WIM system |
| **DL timing precision is 3.5× better** | Both CNN and TCN achieve MATE ≈ 0.24–0.25 samples vs 0.87 for the baseline |
| **TCN is the recommended model** | Equal or better F1 than CNN, 12× smaller checkpoint (2.6 MB vs ~30 MB), converges in ≤ 10 epochs |
| **F1 gap widens at strict tolerances** | At tolerance = 1 sample, baseline drops to ~0.77 while both DL models remain at ~1.0 |
